In [1]:
import numpy as np
from FallbackGen import FallbackGen
from TDECalculator import TDECalculator
import gc

In [2]:
MBH = 1e6
Rp = 25
a = 0.2
N = 5000
E = 1.0
Q = 0.0
orbit="rel"

In [3]:
mass_p = TDECalculator('MAMS2Msun', orbit, MBH, Rp, a, N=N)

In [4]:
sample_p = mass_p.rel_whole_star_sample()

In [5]:
radii_p = sample_p['rr']

rtde = mass_p.R_TDE
Lz = mass_p.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_p <= 0.5,
    rtde - radii_p * mass_p.Rstar,
    rtde + radii_p * mass_p.Rstar
)

deltaE = mass_p.Rstar / mass_p.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_p['dEnergy_random'] * deltaE 
dLz = mass_p.dLz_random
mass_ratio = mass_p.mass_ratio

In [6]:
print(f"Lz = {Lz}")
print(f"dLz range: [{dLz.min():.4e}, {dLz.max():.4e}]")
print(f"total_Lz range: [{(Lz + dLz).min():.4e}, {(Lz + dLz).max():.4e}]")
print(f"dE range: [{dE.min():.4e}, {dE.max():.4e}]")
print(f"total_E range: [{(E + dE).min():.4e}, {(E + dE).max():.4e}]")
print(f"bound fraction: {((E + dE) < 1.0).mean():.3f}")

Lz = 7.354962919731006
dLz range: [-1.2158e+01, 1.2156e+01]
total_Lz range: [-4.8034e+00, 1.9511e+01]
dE range: [-6.2976e-02, 6.2976e-02]
total_E range: [9.3702e-01, 1.0630e+00]
bound fraction: 0.499


In [7]:
dT_p = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, dE, dLz, N)
np.save('dT_p.npy', dT_p.dTs)
gc.collect()

Computing radial periods for 116,280,000 particles ...
  E  range: [0.937024, 1.062976]
  Q  range: [0.000e+00, 0.000e+00]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 11610/11610 (100%)
  Bound: 58,046,820 / 116,280,000
  Valid roots: 57,262,835
  Valid Lambda_r: 57,262,835
  Quadrature: 1146 chunks ...
    chunk 1146/1146  (100%)
  Successful T_r: 57,262,835 / 116,280,000


0

In [8]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample["dEnergy_random"]
    dT_rand = dT / delta
    dMass = whole_star_sample["dMass"]

    bins_E = np.linspace(-2.0, 2.0, 1000)
    bins_T = np.logspace(0.0, 6.0, 1000)

    total_mass = np.sum(dMass)
    if not np.isfinite(total_mass) or total_mass <= 0.0:
        raise ValueError("Sample has non-finite or non-positive total mass")

    valid_E = (
        np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dE_rand >= bins_E[0])
        & (dE_rand <= bins_E[-1])
    )
    mass_E, edges_E = np.histogram(
        dE_rand[valid_E],
        bins=bins_E,
        weights=dMass[valid_E] / total_mass,
        density=False,
    )
    hist_E = mass_E / np.diff(edges_E)

    valid_T = (
        np.isfinite(dT_rand)
        & np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dT_rand >= bins_T[0])
        & (dT_rand <= bins_T[-1])
    )
    mass_T, edges_T = np.histogram(
        dT_rand[valid_T],
        bins=bins_T,
        weights=dMass[valid_T] / total_mass,
        density=False,
    )
    hist_T = mass_T / np.diff(edges_T)

    energy_fraction = np.sum(dMass[valid_E]) / total_mass
    returning_fraction = np.sum(dMass[valid_T]) / total_mass
    assert np.isclose(
        np.sum(hist_E * np.diff(edges_E)), energy_fraction, rtol=1e-12
    )
    assert np.isclose(
        np.sum(hist_T * np.diff(edges_T)), returning_fraction, rtol=1e-12
    )

    print(
        "mass fractions: "
        f"energy range={energy_fraction:.6f}, "
        f"returning/time range={returning_fraction:.6f}"
    )

    energy_plot = {
        "x": 0.5 * (edges_E[:-1] + edges_E[1:]),
        "y": hist_E,
    }
    fallback_plot = {
        "x": 0.5 * (edges_T[:-1] + edges_T[1:]),
        "y": hist_T,
    }
    return energy_plot, fallback_plot

In [9]:
DeltaE = mass_p.Rstar / mass_p.Rp**2
DeltaT = 1 / DeltaE**1.5

In [10]:
rel_p_E, rel_p_T = make_plot_dicts(sample_p, dT_p.dTs, DeltaT)

mass fractions: energy range=1.000000, returning/time range=0.498735


In [11]:
import json

adden = "m2_rp25_a0p2"

with open(f"Fallback_Data/rel_p_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_p_E.items()}, f)
with open(f"Fallback_Data/rel_p_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_p_T.items()}, f)